In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [19]:
import numpy as np

from datasets.mnist import load_mnist
from b1.module.sample import init_network, forward

In [5]:
(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)

# Loss Fuction
- 모델의 예측값과 정답값의 불일치를 하나의 스칼라 값으로 측정하는 함수
- 단일 데이터 포인트에 대해 적용
- 미분을 통해 학습을 위한 지표로 사용 가능 - 학습 데이터에 대한 연속적인 적합도 평가 

## SSE
- sum square error
- 타겟과 출력의 각 피처별 차이 제곱의 합
- $$E = \frac{1}{2} \sum_k(y_k-t_k)^2$$


In [6]:
def sse(y, t):
    return 0.5 * np.sum((y-t)**2)

## CEE
- cross entrophy error
- 타겟에 대한 음의 로그 가능도의 가중합
- 0 방지 - 작은 값을 더해 근사
- one-hot $$E = -\sum_k t_k \log y_k \approx -\sum_k t_k \log (y_k+\delta)$$
- label $$E = -\sum_k \log y_{k, t_k} \approx -\sum_k \log (\log y_{k, t_k} +\delta)$$
### 미니 배치
- $$E = -\frac{1}{N}\sum_n\sum_k t_{nk} \log y_{nk} \approx -\frac{1}{N}\sum_n\sum_k t_{nk} \log(y_{nk}+\delta)$$


In [ ]:
def cross_entropy_error(y, t):
    if y.ndim == 1: # data point -> batch 
        t = t.reshape(1, t.size)
        y = y.reshape(1, y.size)

    batch_size = y.shape[0] 
    return -np.sum(t * np.log(y + 1e-7)) / batch_size                           # one hot target
    # return -np.sum(np.log(y[np.arange(batch_size), t] + 1e-7)) / batch_size   # label target
    # y[np.arange(batch_size), t] - 배치 번째 데이터에 대해 타겟에 해당하는 신경망 출력값을 가져옴

In [ ]:
t = [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
y = [0.1, 0.05, 0.6, 0.0, 0.05, 0.1, 0.0, 0.1, 0.0, 0.0]

print(cross_entropy_error(np.array(y), np.array(t)))


In [31]:
# random batch
train_size = x_train.shape[0]   # train data size
batch_size = 10                 # batch size - hyper pram
batch_mask = np.random.choice(train_size, batch_size)  # random select batch indexes from train data
x_batch = x_train[batch_mask]  # batch data
t_batch = t_train[batch_mask]  # batch label
network = init_network()
y_batch = forward(network, x_batch)
print(cross_entropy_error(y_batch, t_batch))


6.6995421785816704
